In [1]:
import os
import numpy as np
import pandas as pd

chrom       = 22
geno_dir    = "/dbgap/martinzh_ukb16549/ziyanzha/Genotype/UKBimp_337K_MAF001_INFO06"
pvar_path   = f"{geno_dir}/ukb_imp_chr{chrom}_v3.pvar"
region_path = "/dbgap/martinzh_ukb16549/ziyanzha/Annotation/gene_regions_exon_tss1kb_gencode_v19.tsv"
out_dir     = f"{geno_dir}/gxg_gene_table"
min_snps    = 2                                     # a gene needs >= 2 SNPs for pairwise interactions


In [2]:
# skip the "##" meta lines at the top (if any); the "#CHROM ..." line is the header
n_meta = 0
#count how many row should skip
with open(pvar_path) as f:
    for line in f:
        if line.startswith("##"):
            n_meta += 1
        else:
            break

pvar = pd.read_csv(pvar_path, sep="\t", skiprows=n_meta, usecols=[0, 1, 2])
pvar.columns = ["chrom", "pos", "id"]
pvar["snp_idx"] = np.arange(len(pvar))              # 0-based row in the .pvar = column in the genotype matrix

(126417, 4)
sorted by position: True


,chrom,pos,id,snp_idx
0,22,16051249,rs62224609,0
1,22,16053862,rs62224614,1
2,22,16054454,rs7286962,2
3,22,16057417,rs62224618,3
4,22,16430363,rs200408963,4


In [3]:
## read reference and keep the current chr
regions = pd.read_csv(region_path, sep="\t", dtype={"chrom": str})
regions = regions[regions["chrom"] == str(chrom)].reset_index(drop=True)

intervals: 5802
genes: 1263


,chrom,start,end,gene_id,gene_name,gene_type
0,22,16061157,16063236,ENSG00000233866.1,LA16c-4G1.3,lincRNA
1,22,16075172,16077172,ENSG00000229286.1,LA16c-4G1.4,pseudogene
2,22,16083826,16085826,ENSG00000235265.1,LA16c-4G1.5,pseudogene
3,22,16100517,16100648,ENSG00000223875.1,NBEAP3,pseudogene
4,22,16101276,16101474,ENSG00000223875.1,NBEAP3,pseudogene


In [4]:
#find the SNPs in each interval
pos = pvar["pos"].to_numpy()

# index of the first SNP inside each interval, and one past the last SNP inside it
first_snp    = np.searchsorted(pos, regions["start"].to_numpy(), side="left")
last_snp_end = np.searchsorted(pos, regions["end"].to_numpy(),   side="right")
regions["n_snps"] = last_snp_end - first_snp

rows = []
for i in range(len(regions)):
    for j in range(first_snp[i], last_snp_end[i]):
        rows.append((regions.at[i, "gene_id"], regions.at[i, "gene_name"],
                     regions.at[i, "gene_type"], pvar.at[j, "snp_idx"]))
pairs = pd.DataFrame(rows, columns=["gene_id", "gene_name", "gene_type", "snp_idx"])

print("intervals with >= 1 SNP:", (regions["n_snps"] > 0).sum(), "of", len(regions))

intervals with >= 1 SNP: 3240 of 5802
gene-SNP pairs: 20739


,gene_id,gene_name,gene_type,snp_idx
0,ENSG00000236831.1,YME1L1P1,pseudogene,22
1,ENSG00000236831.1,YME1L1P1,pseudogene,23
2,ENSG00000225293.1,ABCD1P4,pseudogene,62
3,ENSG00000225293.1,ABCD1P4,pseudogene,76
4,ENSG00000225293.1,ABCD1P4,pseudogene,79


In [5]:
# safety check: intervals of a gene were merged, so no SNP should appear twice in a gene
n_before = len(pairs)
pairs = pairs.drop_duplicates(["gene_id", "snp_idx"]).reset_index(drop=True)
print("duplicate gene-SNP pairs removed:", n_before - len(pairs))

# SNPs per gene
n_per_gene = pairs.groupby("gene_id")["snp_idx"].size()
keep_genes = n_per_gene[n_per_gene >= min_snps].index
gene_table = pairs[pairs["gene_id"].isin(keep_genes)].sort_values(["gene_id", "snp_idx"]).reset_index(drop=True)

print("genes with >= 1 SNP:", len(n_per_gene))
print(f"genes with >= {min_snps} SNPs:", len(keep_genes))
print("gene-SNP pairs kept:", len(gene_table))
print("distinct SNPs in any gene:", gene_table["snp_idx"].nunique(), "of", len(pvar))
print("\nSNPs per gene:")
print(n_per_gene[keep_genes].describe().round(1))

duplicate gene-SNP pairs removed: 0
genes with >= 1 SNP: 1171
genes with >= 2 SNPs: 1135
gene-SNP pairs kept: 20703
distinct SNPs in any gene: 18925 of 126417

SNPs per gene:
count    1135.0
mean       18.2
std        18.5
min         2.0
25%         7.0
50%        12.0
75%        23.0
max       147.0
Name: snp_idx, dtype: float64


In [6]:
### save the data
out_path = f"{out_dir}/gene_table_chr{chrom}.tsv"
gene_table.to_csv(out_path, sep="\t", index=False)
print("written:", out_path)
print(gene_table.shape)
gene_table.head()

written: /dbgap/martinzh_ukb16549/ziyanzha/Genotype/UKBimp_337K_MAF001_INFO06/gxg_gene_table/gene_table_chr22.tsv
(20703, 4)


,gene_id,gene_name,gene_type,snp_idx
0,ENSG00000008735.10,MAPK8IP2,protein_coding,125795
1,ENSG00000008735.10,MAPK8IP2,protein_coding,125796
2,ENSG00000008735.10,MAPK8IP2,protein_coding,125797
3,ENSG00000008735.10,MAPK8IP2,protein_coding,125798
4,ENSG00000008735.10,MAPK8IP2,protein_coding,125799
